In [ ]:
import polars as pl
from pathlib import Path
from urllib.request import urlopen, Request
from tqdm import tqdm

from python.ai_detection.prepare_dataset import label

In [ ]:

out_dir = Path("/mnt/Fast2T/data/common_crawl/")

urls = (
    pl.read_csv("/mnt/Fast2T/kotlin/2026/data-science-project/data/paths")
    .with_columns(url="https://data.commoncrawl.org/" + pl.col("url"))
    .sample(100)
    .get_column("url")
    .to_list()
)
CHUNK = 1 << 20

for i, url in enumerate(urls):
    dest = out_dir / Path(url).name
    if dest.exists():
        continue

    resp = urlopen(Request(url))
    total = int(resp.headers.get("Content-Length", 0)) or None

    with (
        open(dest, "wb") as f,
        tqdm(
            total=total,
            unit="B",
            unit_scale=True,
            desc=f"[{i + 1}/{len(urls)}] {dest.name[:40]}",
        ) as bar,
    ):
        while chunk := resp.read(CHUNK):
            f.write(chunk)
            bar.update(len(chunk))

In [ ]:
import gzip
import re
from pathlib import Path

import polars as pl
import pyarrow.parquet as pq
from warcio.archiveiterator import ArchiveIterator

_SURROGATES = re.compile(f"[{chr(0xD800)}-{chr(0xDFFF)}]")


def sanitize_utf8(s: str) -> str:
    if not s:
        return s
    s = _SURROGATES.sub("\ufffd", s)
    return s.encode("utf-8", "replace").decode("utf-8")


_charset_re = re.compile(r"charset\s*=\s*([^\s;]+)", re.IGNORECASE)


def charset_from_content_type(ct: str) -> str | None:
    if not ct:
        return None
    m = _charset_re.search(ct)
    return m.group(1).strip("\"'") if m else None


def decode_html(body: bytes, charset: str | None) -> str:
    if not body:
        return ""
    enc = (charset or "utf-8").strip()
    try:
        text = body.decode(enc, errors="replace")
    except (LookupError, UnicodeDecodeError):
        text = body.decode("utf-8", errors="replace")
    return sanitize_utf8(text)


def warc_to_single_parquet_html_only(
        warc_paths: list[Path],
        out_file: Path,
        chunk_size: int = 20_000,
        zstd_level: int | None = None,  # optional
        require_status_200: bool = False  # optional
) -> None:
    out_file.parent.mkdir(parents=True, exist_ok=True)
    if out_file.exists():
        out_file.unlink()  # avoid accidental append/overwrite ambiguity

    contents: list[str] = []
    writer: pq.ParquetWriter | None = None

    def flush() -> None:
        nonlocal contents, writer
        if not contents:
            return

        n = len(contents)
        df = df = pl.DataFrame(
            {
                "content": pl.Series("content", [sanitize_utf8(x) for x in contents], dtype=pl.Utf8),
            }
        )

        table = df.to_arrow()

        if writer is None:
            writer = pq.ParquetWriter(
                where=str(out_file),
                schema=table.schema,
                compression="zstd",
                compression_level=zstd_level,
                use_dictionary=True,
                write_statistics=False
            )

        writer.write_table(table)
        contents = []

    try:
        for warc_path in warc_paths:
            with gzip.open(warc_path, "rb") as f:
                for record in ArchiveIterator(f):
                    if getattr(record, "rec_type", None) != "response":
                        continue

                    hh = getattr(record, "http_headers", None)
                    if not hh:
                        continue

                    ct = (hh.get_header("Content-Type") or "")
                    if "text/html" not in ct.lower():
                        continue

                    if require_status_200:
                        try:
                            if int(hh.get_statuscode() or 0) != 200:
                                continue
                        except Exception:
                            continue

                    body_bytes = record.content_stream().read()
                    html = decode_html(body_bytes, charset_from_content_type(ct))
                    if not html:
                        continue

                    contents.append(sanitize_utf8(html))
                    if len(contents) >= chunk_size:
                        flush()

        flush()
    finally:
        if writer is not None:
            writer.close()


for file in Path("/mnt/data-dump/common_crawl/").iterdir():
    if not file.name.endswith(".warc.gz"):
        continue

    print(file)
    warc_to_single_parquet_html_only(
        warc_paths=[file],
        out_file=Path("/mnt/Fast2T/data/common_data_parquet/") / file.name.replace(".warc.gz", ".parquet"),
    )

In [ ]:
import polars as pl
import pyarrow.parquet as pq
from pathlib import Path
import tqdm

files = Path("/mnt/Fast2T/data/new/processed").glob("*.parquet")
rows = [pq.read_metadata(f).num_rows for f in tqdm.tqdm(files)]

print(rows)
print(sum(rows))
print(sum(rows) / len(rows))

In [ ]:
import polars as pl

ai_type = pl.Enum(["AI", "UNKNOWN", "HUMAN"])

blacklist = [
    "Please wait while your request is being verified...",
    "AI scrapers break the web, to use this page you'll need JavaScript enabled.",
    "Javascript is required for this site. Consider enabling Javascript or upgrading to a modern browser.",
    "This domain name registration has expired and renewal or deletion are pending. If you are the registrant and want to renew the domain name, please contact your registration service provider.",
    "We're sorry but doesn't work properly without JavaScript enabled. Please enable it to continue."
]

(
    pl
    .read_parquet("/mnt/Fast2T/data/new/clean_data/CC-MAIN-20260212024315-20260212054315-00248.parquet")
    .with_columns(
        pl
        .when(pl.col('ai') < 0.005).then(pl.lit("HUMAN"))
        .when(pl.col('ai') > 0.9).then(pl.lit("AI"))
        .otherwise(pl.lit("UNKNOWN"))
        .cast(ai_type).alias("is_ai"),
        pl
        .col("outflow")
        .map_elements(
            lambda s: [{ "url": k, "occurrences": v } for k, v in s.items() if v is not None],
            return_dtype=pl.List(pl.Struct({ "url": pl.String, "occurrences": pl.Int64 })),
        )
    )
    .filter(pl.col("lang_en") > 0.3)
    .filter(~pl.col("text").str.contains_any(blacklist))
    .drop("lang_en", "ai")
)

In [ ]:
import polars as pl

(
    pl
    .scan_parquet("/mnt/Fast2T/data/new/stage_5/*.parquet")
    .explode("stems")
    .unnest("stems")
    .group_by("is_ai", "stem")
    .agg(pl.col("occurrence").sum())
    .sink_parquet("/mnt/Fast2T/data/new/stage_6.parquet")
)

In [ ]:
(
    pl
    .scan_parquet("/mnt/Fast2T/data/new/stage_6.parquet")
    .select("is_ai", "stem", "occurrence")
    .with_columns(
        p=pl.col("occurrence") / pl.col("occurrence").sum().over("is_ai")
    )
    .sink_parquet("/mnt/Fast2T/data/new/stage_6a.parquet")
)

In [ ]:
(
    pl
    .read_parquet("/mnt/Fast2T/data/new/stage_6a.parquet")
    .group_by("is_ai")
    .agg(pl.col("p").sum())
)

In [ ]:
import matplotlib.pyplot as plt

__df = (
    pl
    .scan_parquet("/mnt/Fast2T/data/new/stage_6a.parquet")
    .pivot("is_ai", index="stem", values="p", on_columns=["HUMAN", "UNKNOWN", "AI"])
    .top_k(k=20, by="HUMAN")
    .collect()
)

In [ ]:
import numpy as np

plt.figure(figsize=(20, 10))

x = np.arange(len(__df["stem"]))

plt.bar(x - 0.2, __df["HUMAN"], 0.2, label="HUMAN")
plt.bar(x, __df["UNKNOWN"], 0.2, label="UNKNOWN")
plt.bar(x + 0.2, __df["AI"], 0.2, label="AI")
plt.xticks(x, __df["stem"])
plt.legend()
plt.show()

In [2]:
import polars as pl

(
    pl
    .read_parquet("/mnt/Fast2T/data/new/stage_5/CC-MAIN-20260206181458-20260206211458-00190.parquet")
)

host,url,text,stems,outflow,is_ai
str,str,str,list[struct[2]],list[struct[2]],enum
"""03-128-53-208.plesk.page""","""http://03-128-53-208.plesk.pag…","""Day Two Draft Recap: Cubs Add …","[{""."",57}, {"","",32}, … {""far"",1}]","[{""03-128-53-208.plesk.page"",41}, {""www.facebook.com"",7}, … {""flipboard.com"",1}]","""HUMAN"""
"""550squadronassociation.org.uk""","""http://550squadronassociation.…","""Listed below is the informatio…","[{""."",10}, {"","",8}, … {""2026"",1}]","[{""550squadronassociation.org.uk"",13}]","""UNKNOWN"""
"""acemjmservices.com""","""http://acemjmservices.com/cate…","""March 8, 2023March 8, 2023/ ad…","[{"","",3}, {""8"",2}, … {""write"",1}]","[{""acemjmservices.com"",11}, {""rarathemes.com"",1}, {""wordpress.org"",1}]","""HUMAN"""
"""activewin.com""","""http://activewin.com/mac/comme…","""User Controls New User Login E…","[{""window"",8}, {""."",8}, … {""volum"",1}]","[{""www.activewin.com"",7}, {""media.fastclick.net"",1}, … {""telegra.ph"",120}]","""HUMAN"""
"""advancediesel.net""","""http://advancediesel.net/produ…","""Skip to navigation Skip to con…","[{""−−−"",476}, {""auto"",344}, … {""add"",1}]","[{""advancediesel.net"",869}, {""facebook.com"",1}, {""www.hisdesigns.org"",1}]","""HUMAN"""
…,…,…,…,…,…
"""zilvera.nl""","""https://zilvera.nl/product/squ…","""Description Square amethyst fl…","[{"","",30}, {""."",21}, … {""describ"",1}]","[{""en.wikipedia.org"",1}]","""HUMAN"""
"""zingsphere.co.uk""","""https://zingsphere.co.uk/strat…","""Winning in online casinos isn’…","[{"","",38}, {""."",24}, … {""maxim"",1}]",[],"""AI"""
"""zjlindal.en.made-in-china.com""","""https://zjlindal.en.made-in-ch…","""LED Tri-Proof Light, LED Water…","[{""light"",6}, {"","",5}, … {""."",1}]",[],"""HUMAN"""


In [5]:
import nltk
from nltk.corpus import wordnet

nltk.download('wordnet')
nltk.download('english_wordnet')

len(set(wordnet.words()))

[nltk_data] Downloading package wordnet to /home/qr/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package english_wordnet to
[nltk_data]     /home/qr/nltk_data...
[nltk_data]   Package english_wordnet is already up-to-date!


147306